In [1]:
import pyspark


In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("BaristaCoffeeShop") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/28 00:05:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.sparkContext.setLogLevel("ERROR")

Reading data from csv


In [6]:
file_location = "barista_coffee_sales_data_for_eda.csv"
df1 = spark.read.format('csv').option('header','true').option('inferSchema','true').load(file_location)

In [7]:
df1.show(10)

+----------+-------------------+----------------+---+----------+--------+--------------+-----------+------------+---------------+--------------+------------------+-------------------------+--------------+----------------+------------+-------------+-------------+
|order_date|         order_time|product_category|qty|unit_price|store_id|store_location|customer_id|customer_age|customer_gender|loyalty_member|is_repeat_customer|customer_discovery_source|payment_method|discount_applied|total_amount|promo_applied|   promo_type|
+----------+-------------------+----------------+---+----------+--------+--------------+-----------+------------+---------------+--------------+------------------+-------------------------+--------------+----------------+------------+-------------+-------------+
|2023-04-29|2025-05-28 22:47:20|           Pizza|  3|      7.08| STORE_7|       Airport|     CUST_1|          36|         Female|         false|             false|                  Walk-in|          Cash|       

In [8]:
df1.summary()

DataFrame[summary: string, product_category: string, qty: string, unit_price: string, store_id: string, store_location: string, customer_id: string, customer_age: string, customer_gender: string, customer_discovery_source: string, payment_method: string, discount_applied: string, total_amount: string, promo_type: string]

In [9]:
df1.printSchema()

root
 |-- order_date: date (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- loyalty_member: boolean (nullable = true)
 |-- is_repeat_customer: boolean (nullable = true)
 |-- customer_discovery_source: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- discount_applied: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- promo_applied: boolean (nullable = true)
 |-- promo_type: string (nullable = true)



In [21]:
df1.createOrReplaceTempView("Records")
Analysis1 = spark.sql("Select customer_id,customer_age,loyalty_member,order_date from Records")


In [22]:
Analysis1.createOrReplaceTempView("Analysis1")

In [24]:
# Lets get the count of customers who are in the range between 25-35 differentate by there loyalty membership.
d = spark.sql("Select Count(*) as count_of_customers,loyalty_member from Analysis1 where ((customer_age>=25) and (customer_age<=35)) group by loyalty_member")

In [25]:
d = spark.sql("""
    SELECT 
        COUNT(*) AS count_of_customers,
        loyalty_member 
    FROM Analysis1 
    WHERE customer_age BETWEEN 25 AND 35 
    GROUP BY loyalty_member
""")

In [ ]:
d.show()

+------------------+--------------+
|count_of_customers|loyalty_member|
+------------------+--------------+
|             11731|          true|
|             11606|         false|
+------------------+--------------+



25/05/28 06:38:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE